<a href="https://colab.research.google.com/github/Magar-Bhuwan/ML-Internship-Assignments/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Table(s)

This analysis uses the `fact_content_daily_performance` table from the FlyRank internship warehouse. The table contains daily Search Console and GA4 performance metrics for each content item.

### Excluded

I exclude identifier fields such as `client_hash_id` and `content_hash_id` from the model because they uniquely identify records but do not provide meaningful predictive information.

In [107]:
!pip install -q huggingface_hub datasets

import pandas as pd

from huggingface_hub import login
from google.colab import userdata
from datasets import load_dataset

login(token=userdata.get("HF_TOKEN"))

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train[:5000]"                    # I have to split because my RAM resources couldn't handle this large data
)
df = dataset.to_pandas()
df["report_date"] = pd.to_datetime(df["report_date"])

print(df.info())
df.head()

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 30 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   report_date               5000 non-null   datetime64[ns]
 1   client_hash_id            5000 non-null   object        
 2   content_hash_id           5000 non-null   object        
 3   client_has_gsc            5000 non-null   bool          
 4   client_has_ga4            5000 non-null   bool          
 5   gsc_data_available        5000 non-null   bool          
 6   ga4_data_available        5000 non-null   bool          
 7   gsc_impressions           5000 non-null   int64         
 8   gsc_clicks                5000 non-null   int64         
 9   gsc_sum_position          5000 non-null   int64         
 10  gsc_avg_position          5000 non-null   float64       
 11  ga4_pageviews             5000 non-null   int64         
 12  ga4_sessions        

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


In [108]:
df.columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events']

# Unit of Analysis

One row represents the daily performance of one content item (`content_hash_id`) for one client (`client_hash_id`) on a specific reporting date (`report_date`).

# Table

This analysis uses the `fact_content_daily_performance` table from the FlyRank Internship Warehouse.

# Time Window

The assignment recommends using a mid-panel month (for example March 2026). Because the full warehouse exceeds the memory available in Google Colab Free, I used the first 5,000 rows to demonstrate the workflow.

# Prediction Task

Classification

The objective is to classify whether a content item is likely to become a High Performer or Low Performer using historical Search Console and GA4 metrics.

# Deliberately Excluded

Identifier columns (`client_hash_id`, `content_hash_id`) are excluded because they uniquely identify records but do not provide predictive information.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Feature Columns

- gsc_impressions
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- scroll_events

These columns describe historical search visibility, traffic and engagement and are available before making a prediction.

---

## Label

The dataset does not contain a predefined High Performer label.

Since the warehouse does not provide a predefined business label, I created a proxy label (high_performer) using the median of historical gsc_clicks.


High Performer = gsc_clicks >= median(gsc_clicks)

Low Performer = otherwise.

---

## Context Columns

- report_date
- client_hash_id
- content_hash_id

These describe each observation but are not prediction features.

---

## Excluded Columns

- client_hash_id
- content_hash_id

Reason:

They are identifiers only and would not help the model generalize.

In [109]:
df["high_performer"] = (
    df["gsc_clicks"] >= df["gsc_clicks"].median()
).astype(int)

df[["gsc_clicks","high_performer"]].head()

,gsc_clicks,high_performer
0,0,1
1,0,1
2,0,1
3,0,1
4,0,1


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1

Verify that one row represents one content item on one reporting date.

In [110]:
duplicates = df.duplicated(
    subset=[
        "report_date",
        "client_hash_id",
        "content_hash_id"
    ]
).sum()

print("Duplicate rows:", duplicates)

Duplicate rows: 0


### Query 2

Verify the number of rows and the reporting date range.

In [111]:
print("Rows:", len(df))

print("Start Date:", df["report_date"].min())

print("End Date:", df["report_date"].max())

Rows: 5000
Start Date: 2025-01-27 00:00:00
End Date: 2025-02-12 00:00:00


### Query 3

Verify data availability using the availability flags.

In [112]:
print("Rows with GSC data available")

print("GSC Available:", df["gsc_data_available"].sum())
print("GA4 Available:", df["ga4_data_available"].sum())

print()

print("Rows with GA4 data available")

print(df[df["ga4_data_available"]==True].shape)

Rows with GSC data available
GSC Available: 5000
GA4 Available: 0

Rows with GA4 data available
(0, 31)


In [113]:
available = df[df["gsc_data_available"] == True]

print(len(available))

5000


## Five Features

In [114]:
feature_frame = df[
[
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "scroll_events"
]
]

feature_frame.head()

,gsc_impressions,gsc_avg_position,ga4_pageviews,ga4_sessions,scroll_events
0,30,3.833333,0,0,0
1,5,71.600000,0,0,0
2,1,34.000000,0,0,0
3,6,23.333333,0,0,0
4,5,17.800000,0,0,0


### Why these features are available

1. gsc_impressions

Knowable because historical Search Console impressions already exist before prediction.

2. gsc_avg_position

Knowable because previous search ranking is already recorded.

3. ga4_pageviews

Knowable because historical pageviews have already occurred.

4. ga4_sessions

Knowable because previous sessions exist before prediction.

5. scroll_events

Knowable because user engagement is already measured before prediction.

### Leakage Experiment

In [115]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X = feature_frame

y = df["high_performer"]

X_train,X_test,y_train,y_test=train_test_split(
    X,y,
    test_size=0.2,
    random_state=42
)

model=DecisionTreeClassifier(random_state=42)

model.fit(X_train,y_train)

pred=model.predict(X_test)

print("Honest Accuracy:",accuracy_score(y_test,pred))

Honest Accuracy: 1.0


### Adding Leakage

In [116]:
feature_frame_leak = feature_frame.copy()

feature_frame_leak["leak"]=df["high_performer"]

### Training and Testing the Leakage

In [117]:
X=feature_frame_leak

X_train,X_test,y_train,y_test=train_test_split(
    X,y,
    test_size=0.2,
    random_state=42
)

model=DecisionTreeClassifier(random_state=42)

model.fit(X_train,y_train)

pred=model.predict(X_test)

print("Accuracy WITH leakage:",accuracy_score(y_test,pred))

Accuracy WITH leakage: 1.0


### Removing Leakage

In [118]:
feature_frame_leak = feature_frame_leak.drop(columns=["leak"])

print("Leakage column removed.")

Leakage column removed.


This demonstrates why features derived from the target must never be included during training, as they produce misleadingly high evaluation scores.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limitations

- This analysis uses only historical Search Console and GA4 data.
- The classification label is a proxy created from historical clicks rather than a predefined business label.
- Because the full warehouse exceeded the memory available in Google Colab Free, I used the first 5,000 rows for exploration and demonstration.
- Results from this subset may not fully represent the complete warehouse.
- The analysis supports decision-making but cannot explain why users searched or engaged with the content.

### **Note:**
I used the first 5,000 rows of the dataset because the full `fact_content_daily_performance` table exceeded the memory available in Google Colab Free. This subset was used for exploration and completing the required data contract tasks.

## Self-check

Before you submit, confirm each line honestly:

- ✅ Every section above is filled — markdown thinking AND the code that backs it
- ✅ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✅ No client names, URLs, or private queries anywhere
- ✅ My claims use careful words: observed, measured, directional, decision-support
- ✅ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.